In [1]:
%load_ext autoreload
%autoreload 2

import util as yu
from util import *
import util_moments as yum

yu.setpath('analysis_A20')

enss=['b','c','d','e']
stouts=[5,7,10,13,15,20,25,30,35,40]

In [2]:
def decodeTask(task):
    n2qpp1,ff,j=task.split('_')
    j=j.replace(',',';')
    n2qpp1=tuple([int(ele) for ele in n2qpp1.split(',')])
    return (n2qpp1,ff,j)

selection2key2bare_conn=yu.load_pkl_reg('selection2key2bare_conn')
if selection2key2bare_conn is None:
    path='data_aux/dat_ignore/analysis_A20_conn_run'
    with open(path,'r') as f:
        tasks=f.read().splitlines()
        
    c1s=['all','unequal','equal']; c2s=['err']
    cases_todo=['_'.join([c1,c2]) for c1,c2 in product(c1s,c2s)]

    tfs_phy=[0.6,0.7,0.8,0.9,1.0]
    tcs_phy=[0.15,0.2,0.3,0.4]

    selection2key2bare_conn={}
    for case in cases_todo:
        for task in tasks:
            (n2qpp1,ff,j)=decodeTask(task)
            
            for ens in enss:
                fits=yu.getFits(f'{n2qpp1}_{ff}_{j}_{ens}_{case}_2st')
                if fits is None:
                    continue
                fitlabels=[fit[0] for fit in fits]
                
                for tf_phy,tc_phy in product(tfs_phy,tcs_phy):
                    flag=True
                    
                    ind=np.argmin([np.abs(tfmin-tf_phy/yu.ens2a[ens]) + np.abs(tcmin[0]+tcmin[1] - tc_phy*2/yu.ens2a[ens] ) for tfmin,tcmin in fitlabels])
                    fit=fits[ind]
                    if fit is None:
                        continue
                    
                    selection=(case,(tf_phy,tc_phy))
                    key=(ens,j,n2qpp1)
                    
                    if selection not in selection2key2bare_conn:
                        selection2key2bare_conn[selection]={}
                    selection2key2bare_conn[selection][key]=fit[1][:,0]
                    
    yu.save_pkl_reg('selection2key2bare_conn',selection2key2bare_conn)
    
key2bare_conn_0Q2=yu.load_pkl_reg('key2bare_conn',pathlabel='analysis_avgx_2')

In [3]:
ens2Njk={'b':725,'c':400,'d':493,'e':225}

# RCs
path='data_aux/RCs.pkl'
with open(path,'rb') as f:
    ens2RCs_me=pickle.load(f)
ens2RCs={ens:{} for ens in enss}
for ens in enss:
    for key in ens2RCs_me[ens]:
        if key.endswith('err'):
            continue
        ens2RCs[ens][key]=yu.jackknife_pseudo(ens2RCs_me[ens][key],ens2RCs_me[ens][f'{key}_err']*0+1e-10,ens2Njk[ens])[:,0]
        
path='data_aux/RCs_pre.pkl'
with open(path,'rb') as f:
    ens2RCs_pre_me=pickle.load(f)
ens2RCs_pre={ens:{} for ens in ens2RCs_pre_me.keys()}
for ens in ens2RCs_pre_me.keys():
    for key in ens2RCs_pre_me[ens]:
        if key.endswith('err'):
            continue
        ens2RCs_pre[ens][key]=yu.jackknife_pseudo(ens2RCs_pre_me[ens][key],ens2RCs_pre_me[ens][f'{key}_err']*0+1e-10,ens2Njk[ens])[:,0]

In [4]:
tftcphy=(0.6,0.3)
c1s=['unequal','all','equal']
colors=['r','g','b','orange']

key2bare_conn_extrapolated_selected_converted2unequal={}
for j in ['j+;conn','j-;conn']:
    fig,axs=yu.getFigAxs(len(enss)+1,1+len(c1s)+1,Lrow=4,Lcol=8,sharex='col',sharey=True)
    fig.suptitle([f'{c}={c1}' for c,c1 in zip(colors,c1s)])
    yu.addRowHeader(axs,enss)
    yu.addColHeader(axs,['compare']+c1s+['compare at Q2=0'])

    for ic1,c1 in enumerate(c1s):
        key2bare_conn=selection2key2bare_conn[(f'{c1}_err',tftcphy)]
        for iens,ens in enumerate(enss):
            ax=axs[iens,0]            
            xunit=1; yunit=1
            
            Zqq='Zqq^s' if j in ['j+;conn'] else 'Zqq'
            mn='mu=nu' if c1 in ['all','equal'] else 'mu!=nu'
            Z=ens2RCs[ens][f'{Zqq}({mn})']
            
            n2qpp1s=[key[-1] for key in key2bare_conn.keys() if key[0]==ens and key[1]==j]
            Q2s=np.array([yum.n2qpp12Q2(n2qpp1,ens) for n2qpp1 in n2qpp1s])
            ys=np.array([key2bare_conn[(ens,j,n2qpp1)][:] for n2qpp1 in n2qpp1s]).T

            if c1 in ['all','equal']:
                Q2s=np.concatenate([[0],Q2s])
                ys=np.concatenate([key2bare_conn_0Q2[(ens,j)][:,None],ys],axis=1)
            
            if ic1==0:
                m,e=yu.jackme(key2bare_conn_0Q2[(ens,j)]*ens2RCs[ens][f'{Zqq}(mu=nu)'])
                axs[iens,-1].errorbar(-1,m,e,color='orange',label=yu.un2str(m,e))
            
            color=colors[ic1]
            
            mean,err=yu.jackme(ys*Z[:,None])
            plt_x=(Q2s+(ic1-1)*0.005)*xunit; plt_y=mean*yunit; plt_yerr=err*yunit
            ax.errorbar(plt_x,plt_y,plt_yerr,color=color)
            axs[iens,ic1+1].errorbar(plt_x,plt_y,plt_yerr,color=color)
            axs[-1,ic1+1].errorbar(plt_x,plt_y,plt_yerr,color=colors[iens])
            
            Q2cut=1
            inds=[i for i,Q2 in enumerate(Q2s) if Q2<=Q2cut]
            Q2s=Q2s[inds]
            ys=ys[:,inds]
            
            pars_jk,chi2_jk,Ndof=yu.doFit_dipole(Q2s,ys,corrQ=False)
            
            xs_plt=np.arange(0,Q2cut+1e-5,0.05)
            y_plt=np.array([yu.fitfunc_dipole(xs_plt,*pars) for pars in pars_jk])
            mean,err=yu.jackme(y_plt*Z[:,None])
            plt_x=xs_plt*xunit; plt_y=mean*yunit; plt_yerr=err*yunit
            
            axs[iens,ic1+1].fill_between(plt_x,plt_y-plt_yerr,plt_y+plt_yerr,color=color,alpha=0.2,label=yu.un2str(plt_y[0],plt_yerr[0]))
            axs[iens,ic1+1].legend()
            
            axs[iens,-1].errorbar(ic1,plt_y[0],plt_yerr[0],color=color,label=yu.un2str(plt_y[0],plt_yerr[0]))
            axs[iens,-1].set_xlim([-5,5])
            axs[iens,-1].legend()
            
            if c1=='unequal' and j=='j+;conn':
                t=y_plt[:,0]
                if ens=='e':
                    Njk=460
                    mean,err=yu.jackme(t)
                    t=yu.jackknife_pseudo(mean,err,Njk)[:,0]
                key2bare_conn_extrapolated_selected_converted2unequal[(ens,j)]=t
                
            if c1=='all' and j=='j-;conn':
                factor=ens2RCs_me[ens][f'Zqq(mu=nu)']/ens2RCs_me[ens][f'Zqq(mu!=nu)']
                t=y_plt[:,0]*factor
                if ens=='e':
                    Njk=460
                    mean,err=yu.jackme(t)
                    t=yu.jackknife_pseudo(mean,err,Njk)[:,0]
                key2bare_conn_extrapolated_selected_converted2unequal[(ens,j)]=t
            
    yu.finalizePlot(f'Q2_3ways_{j}')

# disc; Q2 fit

In [5]:
def decodeTask(task):
    n2qpp1,ff,j=task.split('_')
    j=j.replace(',',';')
    n2qpp1=tuple([int(ele) for ele in n2qpp1.split(',')])
    return (n2qpp1,ff,j)

selection2key2bare_disc=yu.load_pkl_reg('selection2key2bare_disc')
if selection2key2bare_disc is None:
    path='data_aux/dat_ignore/analysis_A20_disc_run'
    with open(path,'r') as f:
        tasks=f.read().splitlines()
        
    c1s=['unequal']; c2s=['err']
    cases_todo=['_'.join([c1,c2]) for c1,c2 in product(c1s,c2s)]

    tfs_phy=[0.6,0.7,0.8,0.9,1.0]
    tcs_phy=[0.15,0.2,0.3,0.4]

    selection2key2bare_disc={}
    for case in cases_todo:
        for task in tasks:
            (n2qpp1,ff,j)=decodeTask(task)
            
            for ens in enss:
                fits=yu.getFits(f'{n2qpp1}_{ff}_{j}_{ens}_{case}_const')
                if fits is None:
                    continue
                fitlabels=[fit[0] for fit in fits]
                
                for tf_phy,tc_phy in product(tfs_phy,tcs_phy):
                    flag=True
                    
                    ind=np.argmin([np.abs(tfmin-tf_phy/yu.ens2a[ens]) + np.abs(tcmin[0]+tcmin[1] - tc_phy*2/yu.ens2a[ens] ) for tfmin,tcmin in fitlabels])
                    fit=fits[ind]
                    if fit is None:
                        continue
                    
                    selection=(case,(tf_phy,tc_phy))
                    key=(ens,j,n2qpp1)
                    
                    if selection not in selection2key2bare_disc:
                        selection2key2bare_disc[selection]={}
                    selection2key2bare_disc[selection][key]=fit[1][:,0]
                    
    yu.save_pkl_reg('selection2key2bare_disc',selection2key2bare_disc)
    
# key2bare_disc_0Q2=yu.load_pkl_reg('key2bare_conn',pathlabel='analysis_avgx_2')

In [6]:
tftcphy=(0.6,0.3)
c1s=['unequal']
colors=['r']

key2bare_conn_extrapolated_selected_converted2unequal={}
for j in ['j+;disc','js;disc','jc;disc','jg;stout10']:
    fig,axs=yu.getFigAxs(len(enss),1+len(c1s)+1,Lrow=4,Lcol=8,sharex='col',sharey='row')
    fig.suptitle([f'{c}={c1}' for c,c1 in zip(colors,c1s)])
    yu.addRowHeader(axs,enss)
    yu.addColHeader(axs,['compare']+c1s+['compare at Q2=0'])

    for ic1,c1 in enumerate(c1s):
        key2bare_disc=selection2key2bare_disc[(f'{c1}_err',tftcphy)]
        for iens,ens in enumerate(enss):
            ax=axs[iens,0]            
            xunit=1; yunit=1
            
            Z=np.array([1])
            
            n2qpp1s=[key[-1] for key in key2bare_disc.keys() if key[0]==ens and key[1]==j]
            Q2s=np.array([yum.n2qpp12Q2(n2qpp1,ens) for n2qpp1 in n2qpp1s])
            ys=np.array([key2bare_disc[(ens,j,n2qpp1)][:] for n2qpp1 in n2qpp1s]).T

            # if c1 in ['all','equal']:
            #     Q2s=np.concatenate([[0],Q2s])
            #     ys=np.concatenate([key2bare_disc_0Q2[(ens,j)][:,None],ys],axis=1)
            
            # if ic1==0:
            #     m,e=yu.jackme(key2bare_disc_0Q2[(ens,j)]*ens2RCs[ens][f'{Zqq}(mu=nu)'])
            #     axs[iens,-1].errorbar(-1,m,e,color='orange',label=yu.un2str(m,e))
            
            color=colors[ic1]
            
            mean,err=yu.jackme(ys*Z[:,None])
            plt_x=(Q2s+(ic1-1)*0.005)*xunit; plt_y=mean*yunit; plt_yerr=err*yunit
            ax.errorbar(plt_x,plt_y,plt_yerr,color=color)
            axs[iens,ic1+1].errorbar(plt_x,plt_y,plt_yerr,color=color)
            
            # Q2cut=1
            # inds=[i for i,Q2 in enumerate(Q2s) if Q2<=Q2cut]
            # Q2s=Q2s[inds]
            # ys=ys[:,inds]
            
            # pars_jk,chi2_jk,Ndof=yu.doFit_dipole(Q2s,ys,corrQ=False)
            
            # xs_plt=np.arange(0,Q2cut+1e-5,0.05)
            # y_plt=np.array([yu.fitfunc_dipole(xs_plt,*pars) for pars in pars_jk])
            # mean,err=yu.jackme(y_plt*Z[:,None])
            # plt_x=xs_plt*xunit; plt_y=mean*yunit; plt_yerr=err*yunit
            
            # axs[iens,ic1+1].fill_between(plt_x,plt_y-plt_yerr,plt_y+plt_yerr,color=color,alpha=0.2,label=yu.un2str(plt_y[0],plt_yerr[0]))
            # axs[iens,ic1+1].legend()
            
            # axs[iens,-1].errorbar(ic1,plt_y[0],plt_yerr[0],color=color,label=yu.un2str(plt_y[0],plt_yerr[0]))
            # axs[iens,-1].set_xlim([-5,5])
            # axs[iens,-1].legend()
            
            # if c1=='unequal' and j=='j+;conn':
            #     t=y_plt[:,0]
            #     if ens=='e':
            #         Njk=460
            #         mean,err=yu.jackme(t)
            #         t=yu.jackknife_pseudo(mean,err,Njk)[:,0]
            #     key2bare_conn_extrapolated_selected_converted2unequal[(ens,j)]=t
                
            # if c1=='all' and j=='j-;conn':
            #     factor=ens2RCs_me[ens][f'Zqq(mu=nu)']/ens2RCs_me[ens][f'Zqq(mu!=nu)']
            #     t=y_plt[:,0]*factor
            #     if ens=='e':
            #         Njk=460
            #         mean,err=yu.jackme(t)
            #         t=yu.jackknife_pseudo(mean,err,Njk)[:,0]
            #     key2bare_conn_extrapolated_selected_converted2unequal[(ens,j)]=t
                
            
    yu.finalizePlot(f'Q2_3ways_{j}')

# combine with disc

In [7]:
tftc2key2bare=yu.load_pkl_reg('tftc2key2bare',pathlabel='analysis_avgx_2')
ens2Njk={ens:tftc2key2bare[list(tftc2key2bare.keys())[0]][(ens,'j+;conn')].shape[0] for ens in enss}

for tftc in tftc2key2bare.keys():
    for key in key2bare_conn_extrapolated_selected_converted2unequal.keys():
        tftc2key2bare[tftc][key]=key2bare_conn_extrapolated_selected_converted2unequal[key]

In [8]:
# RCs
path='data_aux/RCs.pkl'
with open(path,'rb') as f:
    ens2RCs_me=pickle.load(f)
ens2RCs={ens:{} for ens in enss}
for ens in enss:
    for key in ens2RCs_me[ens]:
        if key.endswith('err'):
            continue
        ens2RCs[ens][key]=yu.jackknife_pseudo(ens2RCs_me[ens][key],ens2RCs_me[ens][f'{key}_err']*0+1e-10,ens2Njk[ens])[:,0]
        
path='data_aux/RCs_pre.pkl'
with open(path,'rb') as f:
    ens2RCs_pre_me=pickle.load(f)
ens2RCs_pre={ens:{} for ens in ens2RCs_pre_me.keys()}
for ens in ens2RCs_pre_me.keys():
    for key in ens2RCs_pre_me[ens]:
        if key.endswith('err'):
            continue
        ens2RCs_pre[ens][key]=yu.jackknife_pseudo(ens2RCs_pre_me[ens][key],ens2RCs_pre_me[ens][f'{key}_err']*0+1e-10,ens2Njk[ens])[:,0]

In [9]:
tftcs=list(tftc2key2bare.keys()); tftcs.sort()

def run(tftc):
    key2bare=tftc2key2bare[tftc]
    yum.extendBare_avgx(key2bare)
    key2phy=yum.bareRC2phy_avgx_new(key2bare,ens2RCs)
    key2phy_pre=yum.bareRC2phy_avgx_pre_new(key2bare,ens2RCs,ens2RCs_pre)
    return key2phy,key2phy_pre

tasks=tftcs
results=yu.parallelizeTasks(run,tasks,max_workers=13)

tftc2key2phy={}; tftc2key2phy_pre={}
for task,res in zip(tasks,results):
    tftc2key2phy[task],tftc2key2phy_pre[task]=res

26/26 tasks completed          


In [10]:
PDF_SET='NNPDF40_nnlo_as_01180'
PDF_SET='JAM22-PDF_proton_nlo'

j2me=yu.load_pkl(f'pkl/lhapdf/reg_ignore/j2me_{PDF_SET}.pkl')
j2color={'jq':'purple','jg':'cyan','jtot':'grey','ju':'r','jd':'g','js':'b','jc':'orange'}
j2label={'jq':'q','jg':'g','jtot':'N','ju':'u','jd':'d','js':'s','jc':'c'}
j2fmt={'jq':'d','jg':'s','jtot':'o','ju':'^','jd':'v','js':'<','jc':'>'}

def addexp(ax0,ax1):
    ax=ax0
    for j in ['jtot','jq','jg']:
        m,e=j2me[j]
        ax.errorbar(0.0002,m,e,color=j2color[j],mfc='white',fmt=j2fmt[j],markersize=4)
    ax=ax1
    for j in ['ju','jd','js','jc']:
        m,e=j2me[j]
        ax.errorbar(0.0002,m,e,color=j2color[j],mfc='white',fmt=j2fmt[j],markersize=4)

def addexp_span(ax0,ax1):
    ax=ax0
    for j in ['jtot','jq','jg']:
        m,e=j2me[j]
        ax.axhspan(m-e,m+e,color=j2color[j],alpha=0.4)
    ax=ax1
    for j in ['ju','jd','js','jc']:
        m,e=j2me[j]
        ax.axhspan(m-e,m+e,color=j2color[j],alpha=0.4)

In [11]:
tfs=list({tf for tf,tc in tftcs}); tfs.sort()
tcs=list({tc for tf,tc in tftcs}); tcs.sort()

for ce in ['MA','const','linear']:
    for nst in stouts:
        fig,axs=yu.getFigAxs(2,2,sharex=True,sharey='row')
        fig.suptitle(f'{ce}; stout{nst}')
        yu.addColHeader(axs,['pert','nonp (pre, partial)'])

        axs[0,0].set_ylim([0,1.4])
        axs[1,0].set_ylim([-0.1,0.5])

        for icol in [0,1]:
            axs[1,icol].set_xlabel(r'$t_{s,low}^{phy}$')
            dic=[tftc2key2phy,tftc2key2phy_pre][icol]
            addexp_span(axs[0,icol],axs[1,icol])
            for tftc in tftcs:
                tf,tc=tftc
                ax=axs[0,icol]
                for j in ['jtot','jq','jg']:
                    key=(f'a=#_{ce}',f'{j};stout{nst}')
                    mean,err=yu.jackme(dic[tftc][key][:,0])
                    plt_x=(tf+tcs.index(tc)*0.01); plt_y=mean; plt_yerr=err
                    ax.errorbar(plt_x,plt_y,plt_yerr,color=j2color[j],fmt=j2fmt[j])
                ax=axs[1,icol]
                for j in ['ju','jd','js','jc']:
                    key=(f'a=#_{ce}',f'{j};stout{nst}')
                    mean,err=yu.jackme(dic[tftc][key][:,0])
                    plt_x=(tf+tcs.index(tc)*0.01); plt_y=mean; plt_yerr=err
                    ax.errorbar(plt_x,plt_y,plt_yerr,color=j2color[j],fmt=j2fmt[j])
                    
        yu.finalizePlot(f'tftc_dep/{ce};stout{nst}',mkdirQ=True)

# selection

In [17]:
tftc_selected=(0.7,0.3)
yu.save_pkl_reg('key2bare_A20_selected',tftc2key2bare[tftc_selected])

stout=10
yu.save_pkl_reg('key2phy_A20_selected',yum.convert_key2phy_stout(tftc2key2phy[tftc_selected],stout))

In [18]:
stouts_plt=stouts
figs=[]
for ce in ['MA','const','linear']:
    tftc=tftc_selected
    list_dic=[
        {'key2phy':yum.convert_key2phy_stout(tftc2key2phy[tftc],stout),
        'key2phy_pre':yum.convert_key2phy_stout(tftc2key2phy_pre[tftc],stout)
        }
        for stout in stouts_plt
    ]
    fig,axs=yum.makePlot_a2dependence_avgx(list_dic,ce=ce)
    
    for i in range(len(list_dic)):
        addexp(axs[0,i],axs[1,i])
        
    fig.suptitle(f'{ce}')
    yu.addColHeader(axs,[f'stout{stout}' for stout in stouts_plt])
    yu.finalizePlot(closeQ=True)
    figs.append(fig)
    
yu.makePDF('a2dep_stout',figs)

In [19]:
ce='MA'; tftc=tftc_selected; nst=10

list_dic=[
    {'key2phy':yum.convert_key2phy_stout(tftc2key2phy[tftc],nst),
    'key2phy_pre':yum.convert_key2phy_stout(tftc2key2phy_pre[tftc],nst)
    }
]
fig,axs=yum.makePlot_a2dependence_avgx(list_dic)
addexp(axs[0,0],axs[1,0])

yu.finalizePlot(f'a2dep_{ce};{tftc};{nst}')

list_dic=[
    {'key2phy':yum.convert_key2phy_stout(tftc2key2phy_pre[tftc],nst),
    'key2phy_pre':yum.convert_key2phy_stout(tftc2key2phy[tftc],nst)
    }
]
fig,axs=yum.makePlot_a2dependence_avgx(list_dic)
addexp(axs[0,0],axs[1,0])

yu.finalizePlot(f'a2dep_{ce};{tftc};{nst}_pre')

In [20]:
case='avgx'
ce='MA'; tftc=tftc_selected; nst=10
key2phy=yum.convert_key2phy_stout(tftc2key2phy[tftc],nst)

j='jv1'
color='r'
fmt='s'

fig,axs=yu.getFigAxs(1,1,Lrow=4,Lcol=6)
ax=axs[0,0]
ax.set_xlim([0,yum.lat_a2s_plt[-1]])
ax.set_xticks([0,0.003,0.006])
ax.set_ylim([-0.1,0.5])

m,e=j2me[j]
ax.errorbar(0.0002,m,e,color=color,mfc='white',fmt=fmt,markersize=4)

color='r'
caselabel={'avgx':r'\langle \mathrm{x} \rangle', 'B20':r'B20', 'J':r'J'}[case]
mean,err=yu.jackme(key2phy[(f'a=#_{ce}',j)])
x=yum.lat_a2s_plt; ymin=mean-err; ymax=mean+err
ax.plot(x,mean,color=color,linestyle='--',marker='')
ax.fill_between(x, ymin, ymax, color=color, alpha=0.1)

label=rf'${caselabel}_{{u-d}}=$'+yu.un2str(mean[0],err[0],forceResult=1)
for iens,ens in enumerate(enss):
    plt_x=yu.ens2a[ens]**2; plt_y,plt_yerr=yu.jackme(key2phy[(ens,j)])
    ax.errorbar(plt_x,plt_y,plt_yerr,color=color,fmt=fmt,label=label if iens==0 else None)
ax.legend()
yu.finalizePlot(f'a2dep_{ce};{tftc};{nst}_jv1')

In [21]:
def key2phy2df(key2phy):
    js=['jtot','jq','jg','ju','jd','js','jc','jv1']
    enss=['a=#_MA','e','d','c','b']
    columns=['a=0' if ens[:3]=='a=#' else yu.ens2label[ens] for ens in enss]
    t=[[yu.jackme_un2str(key2phy[(ens,j)][:,0] if ens[:3]=='a=#' else key2phy[(ens,j)]) for ens in enss] for j in js]
    t=pd.DataFrame(t,index=js,columns=columns)
    return t

key2phy=yum.convert_key2phy_stout(tftc2key2phy[tftc],nst)
df=key2phy2df(key2phy)
display(df)

,a=0,E112,D96,C80,B64
jtot,0.964(66),1.09(17),0.99(12),0.899(73),0.924(59)
jq,0.571(52),0.73(14),0.635(91),0.461(56),0.556(46)
jg,0.393(27),0.363(63),0.359(60),0.437(34),0.368(25)
ju,0.323(23),0.346(47),0.356(34),0.312(27),0.348(22)
jd,0.158(17),0.180(40),0.190(25),0.143(17),0.164(14)
js,0.057(13),0.114(35),0.057(21),0.017(12),0.0360(90)
jc,0.033(11),0.087(32),0.033(18),-0.0104(96),0.0082(71)
jv1,0.1646(100),0.167(14),0.166(14),0.170(13),0.184(12)
